In [ ]:
import os
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.edge.service import Service
from selenium.webdriver.edge.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
options=Options()
options.add_argument("--start-maximized")
options.add_experimental_option("prefs",{"profile.managed_default_content_settings.images":2})
service=Service("C:/WebDriver/msedgedriver.exe")
driver=webdriver.Edge(service=service,options=options)
def extract_brand_and_name(title):
    parts=title.split()
    if parts and re.match(r"(19|20)\d{2}",parts[0]):
        brand = parts[1] if len(parts)>1 else "Unknown"
        name = " ".join(parts[1:])
    else:
        brand = parts[0] if parts else "Unknown"
        name = title
    return brand,name
def parse_car_text(text):
    year=km=fuel=transmission=emi=None
    year_match=re.search(r"\b(19|20)\d{2}\b",text)
    if year_match:
        year=year_match.group()
    km_match=re.search(r"([\d,.]+)\s?km",text,re.I)
    if km_match:
        km=km_match.group(1)
    fuel_match=re.search(r"(Petrol|Diesel|CNG|Electric|Hybrid)",text,re.I)
    if fuel_match:
        fuel=fuel_match.group()
    if "Manual" in text:
        transmission="Manual"  
    elif "Auto" in text:
        transmission= "Automatic"
    emi_match=re.search(r"EMI\s*₹([\d,]+)",text)
    if emi_match:
        emi=emi_match.group(1)
    return year,km,fuel,transmission,emi
city_name = "Delhi NCR"
base_url= "https://www.cars24.com/buy-used-cars-delhi-ncr/"
TARGET_COUNT =3000
scroll_pause_time = 2
driver.get(base_url)
all_cars_dict={}
print(f"Started scrapping for {TARGET_COUNT} cars in {city_name}...")
while len(all_cars_dict)< TARGET_COUNT:
    car_cards=driver.find_elements(By.XPATH,"//div[contains(@class,'styles_outer__')]")
    for card in car_cards:
        try:
            full_text=card.text
            if full_text in all_cars_dict:
                continue
            title=card.find_element(By.XPATH,".//span[contains(@class,'sc-')]").text
            brand,name = extract_brand_and_name(title)
            try:
                price_text = card.find_element(By.XPATH,".//div[contains(@class,'styles_priceWrap__')]/p[last()]").text
                price=price_text.replace("₹", "").replace(",","").strip()
            except:
                price=None
            year,km,fuel,trans,emi=parse_car_text(full_text)
            all_cars_dict[full_text] = { "brand":brand,
            "name":name,"year":year,"km_driven":km,
            "fuel_type":fuel,"transmission":trans,"price":price,
            "emi":emi,"city":city_name}
        except Exception:
            continue
    print(f"Progress: {len(all_cars_dict)} / {TARGET_COUNT}")
    driver.execute_script("window.scrollBy(0,1500);")
    time.sleep(scroll_pause_time)
    try:
        load_more =driver.find_element(By.XPATH,"//button[contains(text(), 'View More')]")
        driver.execute_script("arguments[0].click();",load_more)
        print("Clicked 'View More' button")
    except:
        pass
driver.quit()
df=pd.DataFrame(list(all_cars_dict.values()))
OUTPUT_FOLDER = r"D:\DA MATERIAL\Used_Cars_Price_Analysis\Data\Raw\Cars24"
os.makedirs(OUTPUT_FOLDER,exist_ok=True)
file_path=os.path.join(OUTPUT_FOLDER,f"cars24_{city_name}_cars_data.csv")
df.to_csv(file_path,index=False)
print(f"Completed! Scrapping {len(df)}: cars.")

Started scrapping for 3000 cars in Delhi NCR...
